In [28]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# 01b_data_quality_data_size.py
# Purpose of Script: Obtain the total size of processed data for the research
# project from the EU DSA Website.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Initialization ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Import Google Drive
#~~~~~~~~~~~~~~~~~~~~~~~~~~
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [29]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Import Libraries
#~~~~~~~~~~~~~~~~~~~~~~~~~~
import requests
import re
import time
import pandas as pd

#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Define Input/Output Paths
#~~~~~~~~~~~~~~~~~~~~~~~~~~
path_out = "/content/drive/MyDrive/Colab Notebooks/hsds/xx_dissertation/03_outputs/"

In [30]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Access EU DSA Website Data ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Initialise Scraping Loop Parameters
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Define Final Output
summary = []

# Define Sample Dates
date_from = "01-01-2025"
date_to = "31-05-2026"

# Define Platform IDs
platforms = {"Facebook": 32, "Instagram": 33, "TikTok": 30,
             "YouTube": 27, "X": 22, "Snapchat": 34, "Whatsapp": 88}

# Define KB-MB-GB Conversion Function
def convert_to_gb(value, unit):
    value = float(value)

    if unit == "TB": return value * 1024
    elif unit == "GB": return value
    elif unit == "MB": return value / 1024
    elif unit == "KB": return value / (1024 ** 2)
    else: return 0

# Define Total Data Size
grand_total = 0


In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Loop Through Each Platform ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# For Each Platform & ID
for platform_name, platform_id in platforms.items():

    # Print Progress
    print(f"\nProcessing {platform_name}...")

    # Initilisation
    platform_total = 0
    page = 1

    # Start Loop - Define Breaks Later
    while True:

        # Define URL to Access
        url = (f"https://transparency.dsa.ec.europa.eu/explore-data/download"
               f"?from_date={date_from}"
               f"&to_date={date_to}"
               f"&platform_id={platform_id}"
               f"&page={page}")

        # Send Get Request
        r = requests.get(url)

        # Sleep when Webpage Overloaded
        if r.status_code == 429:
          print("Rate limited. Waiting 30 seconds...")
          time.sleep(30)
          continue

        # Stop When Webpage Broken
        if r.status_code != 200:
          print(f"Stopped on page {page} (HTTP {r.status_code})")
          break

        # Search HTML for 'CSV' Data
        matches = re.findall(r'csv:&nbsp;([0-9.]+)\s*(GB|MB|KB)', r.text)

        # Isolate Only 'FULL' Files CSV Data
        matches = matches[::2]

        # Stop If No Files Found
        if len(matches) == 0:
            break

        # Initialise Page Number
        page_total = 0

        # Convert KB or MB Amounts to GB - For Final Table
        for value, unit in matches:
            page_total += convert_to_gb(value, unit)

        # Increase Platform Total
        platform_total += page_total

        # Increase Page Number
        page += 1

        # Pause
        time.sleep(2)

    print(f"{platform_name}: {platform_total:.2f} GB ({platform_total/1024:.2f} TB)")

    # Increase Full Total by Platform Total
    grand_total += platform_total

    # Append Platform Data to Export Panda
    summary.append({
    "platform": platform_name,
    "size_gb": platform_total,
    "size_tb": platform_total / 1024})

# Convert Final Summary to Panda
df_summary = pd.DataFrame(summary)

# Generate Overview Row for All Platforms
df_summary.loc[len(df_summary)] = {"platform": "All", "size_gb": grand_total,
                                   "size_tb": grand_total / 1024}

# Format Output
df_summary[["size_gb","size_tb"]] = df_summary[["size_gb","size_tb"]].round(2)

In [36]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Export Results ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
df_summary.to_csv(f"{path_out}01b_dq_1_total_data_processed.csv")